# BruSafe · Notebook 01L — M1 forward model against a published factorial design

*Literature variant of notebook 01. The forward model, the priors and the audits are unchanged. The inference section is replaced by a comparison with the 15-run Box–Behnken design of Gutiérrez-González et al. 2019 (doi 10.1038/s41598-019-53200-7, Table 2), which measured yield as a function of IPTG, temperature and induction time for three recombinant proteins in BL21(DE3). These data cannot calibrate the model, because the model has no temperature axis and the design reports a single yield per run, with replicates only at the centre point. They do test its most consequential qualitative claim: that above ~0.1 mM, IPTG dose is the weakest of the three factors.*

*Section numbering follows the CAPSID framework document; this notebook implements Sections 2.1–2.6 and 2.11–2.20.*

**iGEM Tec-Chihuahua 2026 · CAPSID framework**

This notebook uses the expression arm of the pipeline, from IPTG addition to nascent coat protein (CP).

## Workflow

1. **M1 as a coupled ODE system.** The metabolic burden of CP synthesis enters as a feedback term inside the cascade.
2. **Literature-derived priors and a prior predictive simulation**, which give a yield forecast before any experiment.
3. **Identifiability audit**, including one exact structural degeneracy of the model.
4. **Comparison with measured data** (§5): the ranking of IPTG, temperature and induction time predicted by the model is compared with the published design. In notebook 01 this position is occupied by the calibration on synthetic data.

---
## 1 · Setup

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from numba import njit
from scipy.integrate import solve_ivp
import pymc as pm, pytensor.tensor as pt, arviz as az
from pytensor.compile.ops import as_op

RNG = np.random.default_rng(2026)
plt.rcParams.update({"figure.dpi":110,"font.size":9,"axes.grid":True,"grid.alpha":.25,
                     "axes.spines.top":False,"axes.spines.right":False})
print("pymc", pm.__version__, "| arviz", az.__version__)

# ---- Figure style (team palette) -------------------------------------------
# The colours are sampled from the team wordmark. Figures refer to the names
# below instead of hex codes, so the whole set can be restyled from this cell.
# SF Pro is used where installed; matplotlib substitutes another font otherwise.
CREAM, PALE, STEEL = "#F4F3EF", "#CCD8E2", "#89A1C2"
SKY,   BLUE, MID   = "#4CA3DF", "#3E77E1", "#2C5BA5"
NAVY,  INK         = "#173779", "#122047"
from matplotlib.colors import LinearSegmentedColormap
BRAND_CMAP = LinearSegmentedColormap.from_list("tecchihuahua",
                                               [NAVY, MID, BLUE, SKY])
plt.rcParams.update({
    "font.family": ["SF Pro Text", "SF Pro Display", "SF Pro", "Helvetica Neue"],
    "mathtext.fontset": "custom", "mathtext.default": "regular",
    "mathtext.rm": "SF Pro Text", "mathtext.it": "SF Pro Text:italic",
    "mathtext.bf": "SF Pro Text:bold",
    "figure.facecolor": CREAM, "axes.facecolor": CREAM, "savefig.facecolor": CREAM,
    "text.color": INK, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.edgecolor": INK, "xtick.color": INK, "ytick.color": INK,
    "grid.color": STEEL, "grid.alpha": 0.35,
    "legend.facecolor": CREAM, "legend.edgecolor": STEEL, "legend.framealpha": 0.9,
    "axes.prop_cycle": plt.cycler(color=[NAVY, BLUE, SKY, STEEL, MID, PALE]),
})


---
## 2 · The model

### 2.1 · State variables

| Symbol | Meaning | Units |
|---|---|---|
| `X` | biomass | OD₆₀₀ |
| `I` | intracellular inducer | μM |
| `R̃` | active T7 RNA polymerase | normalised 0–1 |
| `m̃` | CP transcript | normalised 0–1 |
| `P` | cumulative CP monomer | µM (cytoplasmic) |

### 2.2 · Equations

$$\phi_h = P/P_{ref}, \qquad \psi = \max\!\left(0,\ 1-\phi_h/\phi_{crit}\right)$$

$$\mu = \mu_{max}\max\!\left(0,1-\tfrac{X}{X_{max}}\right)\psi, \qquad \frac{dX}{dt}=\mu X$$

$$\frac{dI}{dt}=P_d(I_e-I)-\mu I, \qquad f(I)=\varepsilon+(1-\varepsilon)\frac{I^h}{K_I^h+I^h}$$

$$\frac{d\tilde R}{dt}=k_R(f-\tilde R),\qquad
\frac{d\tilde m}{dt}=k_m(\tilde R-\tilde m),\qquad
\frac{dP}{dt}=G\,\tilde m\,\psi-\mu P$$

(The code below adds a first-order loss term, `k_deg`, to the equation for `P`; the reason is given in §3.1.)

**(a) A single capacity factor ψ gates both growth and synthesis (M0).**
Under proteome-partitioning growth laws, expressing a non-functional protein displaces ribosomal
and metabolic sectors. The displacement that slows growth also limits further synthesis, so ψ
appears in both equations. As a result `P` saturates after growth arrests without an explicit cap.

**(b) The cascade is normalised and its gain is lumped into `G`.**
The physical chain (RNAP synthesis × plasmid copy number × transcription rate × translation rate)
is a product of four quantities that protein measurements cannot separate. R̃ and m̃ therefore
carry only the shape of the response, that is, the delay; all of the amplitude is in `G` (see §4).

**(c) Passive-only transport presumes a *lacY*⁻ strain.**
LacY is itself lac-controlled, so in *lacY*⁺ strains such as BL21(DE3) inducer uptake is
autocatalytic and the population becomes bimodal at sub-saturating IPTG. A deterministic mean then
corresponds to no real cell state, since part of the population is fully induced and part is off.
The model therefore applies to a *lacY*⁻ host such as Tuner(DE3), or to doses above roughly
100–200 µM. `I_e` is constant because IPTG is gratuitous. The model does not apply to lactose
induction, which additionally requires hydrolysis, allolactose formation and catabolite repression
(see notebook 02).

In [ ]:
M_CP  = 13.7      # kDa, MS2 coat protein monomer (Golmohammadi et al. 1993)
P_REF = 18000.0   # [M1.4] uM total cytoplasmic protein in CP-mass equivalents (~250 mg/mL)

# ---- Unit convention -------------------------------------------------------
# All internal quantities are in uM and minutes. mM appears only at the plotting
# and design interface, through the two helpers below; the two systems should
# not be mixed.
mM_to_uM = lambda x: np.asarray(x, float)*1e3
uM_to_mM = lambda x: np.asarray(x, float)/1e3
T_END    = 1440.0        # min. forecast horizon (24 h); an earlier version used 300 min (5 h)
# ----------------------------------------------------------------------------

PNAMES = ["mu_max","Xmax","phi_crit","P_d","K_I","h","eps","k_R","k_m","G","k_deg"]

@njit(fastmath=True)
def _rhs(t, Y, p, I_e):
    n = I_e.size; out = np.empty(5*n)
    mu_max, Xmax, phi_crit, P_d = p[0], p[1], p[2], p[3]
    K_I, h, eps, k_R, k_m, G    = p[4], p[5], p[6], p[7], p[8], p[9]
    k_deg = p[10]                              # [M1.18] first-order CP loss
    Kh = K_I**h
    for i in range(n):
        X=Y[i]; I=Y[n+i]; R=Y[2*n+i]; m=Y[3*n+i]; P=Y[4*n+i]
        psi = 1.0 - (P/P_REF)/phi_crit          # [M1.2] proteome-partition growth law
        if psi < 0.0: psi = 0.0
        gx = 1.0 - X/Xmax                       # [M1.1] carrying capacity (phenomenological)
        if gx < 0.0: gx = 0.0
        mu = mu_max*gx*psi
        Ic = I if I > 0.0 else 0.0
        f  = eps + (1.0-eps)*Ic**h/(Kh+Ic**h)   # [M1.9] Hill de-repression  [M1.10] basal leak
        out[i]     = mu*X                       # [M1.1] logistic growth
        out[n+i]   = P_d*(I_e[i]-I) - mu*I      # [M1.5] passive uptake  [M1.8] dilution
        out[2*n+i] = k_R*(f-R)                  # [M1.11] RNAP relaxation
        out[3*n+i] = k_m*(R-m)                  # [M1.12] transcript turnover
        out[4*n+i] = G*m*psi - (mu + k_deg)*P     # [M1.16] gain  [M1.3] gating  [M1.18] loss
    return out

def simulate(theta, doses, t_eval, X0=0.4):
    # All doses are integrated as one augmented system, about 5x faster than looping.
    p = np.asarray(theta,float); I_e = np.asarray(doses,float)
    te = np.asarray(t_eval,float); n = I_e.size
    Y0 = np.concatenate([np.full(n,X0), np.zeros(4*n)])
    # rtol/atol are tighter than the 1e-6/1e-8 used earlier: over a 24-48 h horizon
    # the clip at psi = 0 is a kink, and the solver otherwise accumulates error across it.
    s = solve_ivp(_rhs,(0.0,float(te[-1])),Y0,t_eval=te,args=(p,I_e),
                  method="LSODA",rtol=1e-8,atol=1e-10)
    if (not s.success) or s.y.shape[1]!=te.size: return None
    Y=s.y
    return {"X":Y[0:n],"I":Y[n:2*n],"R":Y[2*n:3*n],"m":Y[3*n:4*n],"P":Y[4*n:5*n]}

def logspan(t_end, n=140, t0=2.0):
    # Log-spaced time grid. The transcript transient lasts minutes and the protein
    # plateau hours, so a linear grid would place almost all points on the plateau.
    return np.concatenate([[0.0], np.logspace(np.log10(t0), np.log10(t_end), n-1)])

#                mu_max Xmax phi_c P_d  K_I   h   eps   k_R   k_m    G     k_deg
_th0 = np.array([0.021,  6., 0.25, 0.35, 30., 2., 0.004, 0.07, 0.30, 95., 0.0035])
simulate(_th0, np.array([25.]), np.array([0.,60.]))   # triggers JIT compilation
import time
_t=time.time()
for _ in range(200): simulate(_th0, np.array([10.,25.,50.,200.]), np.linspace(0,T_END,6))
print(f"forward model: {(time.time()-_t)/200*1000:.2f} ms per evaluation")

### 2.3 · Term-by-term provenance

Every term in `_rhs` carries a tag `[Mx.y]` that indexes the table below. The *Evidence* column
records how each term is supported:

| Class | Meaning |
|---|---|
| **Direct** | The cited work reports this functional form or numerical value |
| **Adapted** | The literature establishes the mechanism; the specific functional form is ours |
| **Assumption** | A modelling choice with no literature claim behind it, justified by argument rather than citation |
| **Reparam.** | A mathematical restatement, not a physical claim; it carries no empirical content |

Classifying a term as an assumption is a statement about its status and not a weakness of the
model. The error to avoid is presenting an assumption as a direct result.

In [ ]:
CITATIONS = pd.DataFrame([
 ("M1.1","dX/dt = mu*X; (1 - X/Xmax)","Logistic carrying capacity stands in for unmodelled nutrient depletion",
  "Assumption","Monod 1949 Annu Rev Microbiol 3:371 gives the mechanistic (substrate-explicit) alternative"),
 ("M1.2","psi = 1 - phi_h/phi_crit","Growth rate falls ~linearly with the mass fraction of non-functional protein",
  "Direct","Scott et al. 2010 Science 330:1099; Klumpp, Zhang & Hwa 2009 Cell 139:1366"),
 ("M1.3","psi also multiplies synthesis","Growth and heterologous expression draw on one shared ribosomal pool",
  "Adapted","Weisse et al. 2015 PNAS 112:E1038. The shared-factor FORM is ours -- see note below"),
 ("M1.4","P_REF = 18000 uM","Cytoplasmic protein ~200-300 mg/mL, expressed in CP-mass equivalents",
  "Direct","Bremer & Dennis 2008 EcoSal Plus; Milo et al. 2010 NAR 38:D750 (BioNumbers)"),
 ("M1.5","P_d*(I_e - I)","IPTG enters independently of LacY at higher concentrations",
  "Direct","Marbach & Bettenbrock 2012 J Biotechnol 157:82"),
 ("M1.6","LacY term OMITTED","lacY-minus strain removes the saturable carrier and its positive feedback",
  "Direct","Marbach & Bettenbrock 2012 (bimodality at low IPTG in lacY+); Ozbudak et al. 2004 Nature 427:737"),
 ("M1.7","I_e held constant","IPTG is gratuitous -- not metabolised, so external pool is not depleted",
  "Direct","Marbach & Bettenbrock 2012. FAILS for lactose -- see M1.7 note"),
 ("M1.8","-mu*I dilution","Volume expansion dilutes every intracellular species",
  "Reparam.","Standard mass balance; no empirical claim"),
 ("M1.9","Hill f(I), h ~ 2","Cooperative inducer binding to LacI; two IPTG sites per dimer",
  "Adapted","Lewis et al. 1996 Science 271:1247 (structure); Yildirim & Mackey 2003 Biophys J 84:2841 (n=2 in model)"),
 ("M1.10","eps (basal leak)","lacUV5 in the DE3 lysogen is leaky; magnitude is construct-dependent",
  "Direct","Studier & Moffatt 1986 JMB 189:113; Dubendorff & Studier 1991 JMB 219:45"),
 ("M1.11","k_R*(f - R)","First-order relaxation lumps RNAP synthesis, folding and dilution into one constant",
  "Assumption","No literature reports this lumped constant. It is a time-scale placeholder."),
 ("M1.12","k_m*(R - m)","Transcript turnover; E. coli median mRNA half-life ~5 min",
  "Direct","Bernstein et al. 2002 PNAS 99:9697 (median 5 min); Selinger et al. 2003 Genome Res 13:216 (6.8 min)"),
 ("M1.13","elongation NOT modelled","T7 RNAP elongates ~230 nt/s at 37 C, ~5x E. coli RNAP -- transcript completion is fast vs all other time scales",
  "Direct","Iost, Guillerez & Dreyfus 1992 J Bacteriol 174:619; Vogel & Jensen 1994 J Bacteriol 176:2807 (E. coli 28-89 nt/s)"),
 ("M1.14","shared RNAP between cassettes","Two T7 promoters compete for one polymerase pool",
  "Adapted","Klumpp & Hwa 2008 PNAS 105:20245 (RNAP partitioning). Not implemented in M1; see §2 note (b)"),
 ("M1.15","CP represses cargo translation","MS2 CP dimer binding the operator blocks downstream translation",
  "Direct","Bernardi & Spahr 1972 PNAS 69:3033; Peabody 1993 EMBO J 12:595. Include ONLY if the pac array is 5' of the epitope ORF"),
 ("M1.16","G (lumped gain)","Product of RNAP level, plasmid copy no., k_tx and k_tl -- not separable from protein data",
  "Reparam.","No citation possible or appropriate. See identifiability audit, §4.2"),
 ("M1.17","ribosome saturation","Translation capacity saturates as transcript load rises",
  "Adapted","Scott et al. 2010; Klumpp et al. 2009. Absorbed into G at this level of detail"),
 ("M1.18","-(mu + k_deg)*P","First-order CP loss lumping proteolysis, inclusion-body partition and leakage. "
  "Without it dP/dt loses every sink once mu -> 0, and P clamps at phi_crit*P_REF for all doses",
  "Assumption","Maurizi 1992 Experientia 48:178 gives ClpP/Lon turnover ranges; no MS2-CP value exists. "
  "Measure it directly with a translation-halt chase -- see §3.1"),
 ("M1.19","log-normal gel error","Band-intensity error is multiplicative at high signal, additive at low",
  "Adapted","Rocke & Durbin 2001 J Comput Biol 8:557 (two-component intensity error model)"),
 ("M1.20","per-gel alpha","Loading, transfer and exposure vary between gels: a batch effect",
  "Adapted","Gelman & Hill 2007, Data Analysis Using Regression and Multilevel Models, ch. 12"),
 ("M1.21","discrepancy term","Absorbs structural model error instead of distorting kinetic parameters",
  "Direct","Kennedy & O'Hagan 2001 JRSS-B 63:425. NOT yet implemented -- add if §10 coverage fails"),
 ("M1.22","SMC sampler","Tempered sequential importance sampling; gradient-free, handles multimodality",
  "Direct","Del Moral, Doucet & Jasra 2006 JRSS-B 68:411"),
 ("M1.23","identifiability audit","Structural and practical identifiability must precede inference",
  "Direct","Raue et al. 2009 Bioinformatics 25:1923"),
 ("M1.24","k_R <-> k_m symmetry","Cascade transfer function is symmetric in its two poles",
  "Reparam.","Derived in §4.1, not cited -- it is a provable property of the model, verified numerically"),
], columns=["tag","term","basis","evidence","reference"])

pd.set_option("display.max_colwidth", 110)
print(CITATIONS.evidence.value_counts().to_string(), "\n")
CITATIONS

> **Note on M1.3.** That growth rate declines with heterologous expression is well established
> (M1.2). That the same factor also scales further synthesis is a modelling choice of ours. It is
> motivated by shared ribosome allocation, but it has not been published in this form. It matters
> because it is what makes `P` saturate.
>
> The term is treated as an assumption. It could be tested by measuring ribosome content (or rRNA)
> across the induction series and checking whether it tracks ψ.

In [ ]:
def mass_balance_residual(theta, doses, t_end=None):
    # Integrity check: CP synthesised = CP accumulated + dilution + degradation losses.
    t_end = T_END if t_end is None else t_end
    te = np.linspace(0,t_end,1201); o = simulate(theta,doses,te); p = np.asarray(theta,float)
    psi = np.clip(1.0-(o["P"]/P_REF)/p[2],0,None)
    mu  = p[0]*np.clip(1.0-o["X"]/p[1],0,None)*psi
    made = np.trapezoid(p[9]*o["m"]*psi, te, axis=1)
    lost = np.trapezoid((mu + p[10])*o["P"], te, axis=1)   # [M1.18] loss now in the ledger
    return np.abs(o["P"][:,-1]-(made-lost))/np.maximum(made,1e-9)

# checked at the LONG horizon and across the full uM->mM span, not just 300 min
for _te in (300.0, T_END, 2880.0):
    r = mass_balance_residual(_th0, np.array([10.,50.,200.,1000.,10000.]), _te)
    print(f"t_end={_te/60:5.1f} h  max relative residual = {r.max():.2e}")
    assert r.max() < 1e-3, "MASS BALANCE VIOLATED -- stoichiometry bug"
print("PASS")

---
## 3 · Literature priors and the pre-experimental forecast

The priors are deliberately wider than the literature point estimates would suggest. A prior that
is too tight conflicts with the data and moves the disagreement into the discrepancy term.

In [ ]:
PRIORS = pd.DataFrame([
 ("mu_max","TruncN",0.021,0.005,"M1.1","Doubling ~33 min, rich medium 37 C",
  "Bremer & Dennis 2008 EcoSal Plus","Direct"),
 ("Xmax","TruncN",6.0,1.5,"M1.1","Shake-flask carrying capacity",
  "Measured with our own negative control","Assumption"),
 ("phi_crit","TruncN",0.25,0.08,"M1.2","Max heterologous proteome fraction; strong T7 reaches 20-30%",
  "Studier & Moffatt 1986 JMB 189:113; Scott et al. 2010 Science 330:1099","Direct"),
 ("P_d","LogN",0.35,0.6,"M1.5","Passive IPTG entry, equilibrates within minutes",
  "Marbach & Bettenbrock 2012 J Biotechnol 157:82","Adapted"),
 ("K_I","LogN",30.0,0.7,"M1.9","Half-maximal IPTG, lacY-minus background",
  "Marbach & Bettenbrock 2012 J Biotechnol 157:82","Direct"),
 ("h","TruncN",2.0,0.5,"M1.9","Two inducer sites per LacI dimer",
  "Lewis et al. 1996 Science 271:1247; Yildirim & Mackey 2003 Biophys J 84:2841","Adapted"),
 ("eps","LogN",0.004,0.8,"M1.10","lacUV5 basal leak in DE3; strongly construct-dependent",
  "Studier & Moffatt 1986; Dubendorff & Studier 1991 JMB 219:45","Adapted"),
 ("k_R","LogN",0.07,0.6,"M1.11","T7 RNAP accumulation, ~15 min time constant",
  "Order-of-magnitude from Studier & Moffatt 1986; the lumped constant is not itself reported","Assumption"),
 ("k_m","LogN",0.30,0.6,"M1.12","Transcript turnover; median E. coli mRNA half-life ~5 min",
  "Bernstein et al. 2002 PNAS 99:9697; Selinger et al. 2003 Genome Res 13:216","Direct"),
 ("G","LogN",95.0,0.8,"M1.16","LUMPED GAIN: RNAP x copy no. x k_tx x k_tl",
  "No citation possible -- centred to give ~20% proteome fraction at saturation","Reparam."),
 ("k_deg","LogN",0.0035,0.9,"M1.18","Lumped CP loss: proteolysis + IB partition + leakage. "
  "t_1/2 ~ 3 h. UNCONSTRAINED by a 5 h experiment -- see the horizon audit in §3.1",
  "Maurizi 1992 Experientia 48:178 (ClpP/Lon ranges); no MS2-CP-specific value exists","Assumption"),
], columns=["param","dist","loc","scale","tag","basis","reference","evidence"])
BOUNDS = {"mu_max":(0.005,0.045),"Xmax":(2.,12.),"phi_crit":(0.05,0.50),"h":(1.,4.)}
pd.set_option("display.max_colwidth",78)
print("Prior provenance by evidence class:")
print(PRIORS.evidence.value_counts().to_string(), "\n")
PRIORS

In [ ]:
def sample_prior(n, rng=RNG):
    out={}
    for _,r in PRIORS.iterrows():
        if r["dist"]=="LogN":
            out[r["param"]] = rng.lognormal(np.log(r["loc"]), r["scale"], n)
        else:
            lo,hi = BOUNDS[r["param"]]; v = rng.normal(r["loc"], r["scale"], n)
            while np.any(bad:=(v<lo)|(v>hi)): v[bad]=rng.normal(r["loc"],r["scale"],bad.sum())
            out[r["param"]]=v
    return pd.DataFrame(out)[PNAMES]

# ---- Dose design -----------------------------------------------------------
# The calibration ladder must straddle K_I (prior median 30 uM), otherwise K_I
# and h are not identifiable (see the saturation audit two cells down). The
# 1 mM rung ties the ladder to the standard induction protocol; it carries almost
# no information about K_I, but it is the condition most commonly used.
DOSES = np.array([5., 15., 50., 150., 500., 1000.])   # uM IPTG (experiment V-A)

# Sampling times: roughly log-spaced and extended past the plateau. The two late
# points are the only data that can constrain k_deg.
TOBS  = np.array([0., 20., 45., 90., 180., 300., 720., 1440.])   # min post-induction

PHI_CYT, GDCW_PER_OD = 1.8e-3, 0.33
yield_mgL = lambda P_uM, OD: P_uM*1e-6*M_CP*1e3*PHI_CYT*(OD*GDCW_PER_OD)*1e3

# Forecast scan: 1 uM to 10 mM, log-spaced. Linear spacing over 0-10 mM would put
# about 90% of the points inside the saturated plateau.
dscan = np.logspace(0, 4, 30)                          # uM
t_fine = logspan(T_END, 160)

prior_df = sample_prior(400)
Pp,ODp = [],[]
for _,row in prior_df.iterrows():
    o = simulate(row.values, DOSES, t_fine)
    if o is not None: Pp.append(o["P"]); ODp.append(o["X"])
Pp,ODp = np.array(Pp),np.array(ODp); Ypp = yield_mgL(Pp,ODp)
print(f"{Pp.shape[0]}/400 prior draws integrated over {T_END/60:.0f} h")
print("doses (mM):", np.array2string(uM_to_mM(DOSES), precision=3))

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(12.5,3.5)); qs=[5,25,50,75,95]
for j,d in enumerate(DOSES):
    q=np.percentile(Ypp[:,j,:],qs,axis=0); c=BRAND_CMAP(j/(len(DOSES)-1))
    ax[0].fill_between(t_fine/60,q[0],q[4],alpha=.13,color=c)
    ax[0].plot(t_fine/60,q[2],color=c,lw=1.8,label=f"{uM_to_mM(d):.3g}")
    qo=np.percentile(ODp[:,j,:],qs,axis=0)
    ax[1].fill_between(t_fine/60,qo[0],qo[4],alpha=.13,color=c); ax[1].plot(t_fine/60,qo[2],color=c,lw=1.8)
ax[0].set(xlabel="time post-induction (h)",ylabel="CP yield (mg/L)",title="Prior predictive: yield")
ax[0].legend(fontsize=7,title="IPTG (mM)")
ax[1].set(xlabel="time post-induction (h)",ylabel="OD$_{600}$",title="Prior predictive: growth (burden)")

Yd=[]
for _,row in prior_df.iloc[:200].iterrows():
    o=simulate(row.values,dscan,np.array([0.,T_END]))
    if o is not None: Yd.append(yield_mgL(o["P"][:,-1],o["X"][:,-1]))
Yd=np.array(Yd); qd=np.percentile(Yd,qs,axis=0)
ax[2].fill_between(uM_to_mM(dscan),qd[0],qd[4],alpha=.18,color=MID)
ax[2].fill_between(uM_to_mM(dscan),qd[1],qd[3],alpha=.28,color=MID)
ax[2].plot(uM_to_mM(dscan),qd[2],color=MID,lw=2)
for d in DOSES: ax[2].axvline(uM_to_mM(d),color=INK,lw=.6,alpha=.35)
ax[2].set(xscale="log",xlabel="IPTG (mM)",ylabel=f"CP yield at {T_END/60:.0f} h (mg/L)",
          title="Predicted dose-response")
plt.tight_layout(); plt.show()
print(f"Prior-predicted optimum ~{uM_to_mM(dscan[np.argmax(qd[2])]):.3g} mM "
      f"({qd[2].max():.0f} mg/L) vs {qd[2][-1]:.0f} mg/L at 10 mM")

### 3.1 · Saturation and horizon audit

Two failure modes appear once the forecast is extended to the mM range and to 24 h. Neither raises
an error: the code runs, the plots look reasonable, and the numbers are wrong. The cell below makes
both visible.

In [ ]:
# ---- A. how much signal survives above K_I? --------------------------------
_probe = np.array([250., 500., 1000., 2000., 5000., 10000.])   # uM
_P = simulate(_th0, _probe, np.array([0., T_END]))["P"][:,-1]
_hi = _P[_probe >= 1000.]
_sig = (_hi.max()-_hi.min())/_hi.mean()
print("A. dose separation in the mM range")
for d,pp in zip(_probe,_P): print(f"     {uM_to_mM(d):6.2f} mM -> P = {pp:9.3f} uM")
print(f"   relative spread over 1-10 mM : {100*_sig:.4f} %")
print(f"   densitometry noise (sig_cp)  : 12 %")
print(f"   -> SNR = {_sig/0.12:.1e}. Doses above ~{5*_th0[4]:.0f} uM are one condition, not several.\n")

# ---- B. is K_I identifiable from an all-mM ladder? -------------------------
print("B. K_I identifiability, all-mM design vs. straddling design")
for _KI in [5., 30., 150., 500.]:
    _th = _th0.copy(); _th[4] = _KI
    _a = simulate(_th, np.array([1e3,5e3,1e4]), np.array([0.,T_END]))["P"][:,-1]
    _b = simulate(_th, DOSES, np.array([0.,T_END]))["P"][:,-1]
    print(f"   K_I={_KI:6.1f}  all-mM: {np.array2string(_a,precision=1)}"
          f"   straddling: {np.array2string(_b,precision=1)}")
print("   all-mM rows are near-identical -> flat posterior, no shrinkage vs prior.\n")

# ---- C. does the long-horizon forecast still depend on dose? ---------------
print(f"C. horizon audit (k_deg = {_th0[10]:.4f} /min, t_1/2 = {np.log(2)/_th0[10]/60:.1f} h)")
_ds = np.array([5.,15.,50.,150.,500.,1000.,10000.])
for _te in [300., 720., 1440., 2880.]:
    for _kP,_lbl in [(0.0,"k_deg=0  "), (_th0[10],"k_deg>0  ")]:
        _th = _th0.copy(); _th[10] = _kP
        _p = simulate(_th,_ds,np.array([0.,_te]))["P"][:,-1]
        print(f"   {_lbl} t={_te/60:5.1f} h  spread across doses = "
              f"{100*(_p.max()-_p.min())/_p.mean():6.1f} %"
              f"   ceiling hit: {np.sum(_p > 0.995*_th[2]*P_REF)}/{_ds.size}")
print("   With k_deg=0 the only brake on P is psi, so every dose converges on")
print(f"   phi_crit*P_REF = {_th0[2]*P_REF:.0f} uM and the dose-response flattens to noise.")

**Interpretation.**

*Checks A and B* concern experimental design. Doses in the mM range behave as a single condition.
Calibrating only there returns the prior on `K_I` and `h`, with an acceptable-looking ESS and no
shrinkage: the sampler works, but the data contain no information about those parameters.
Forecasting at mM doses is appropriate, since that is what the wet lab runs; calibrating only at
mM doses is not. `DOSES` straddles `K_I` for this reason.

*Check C* reveals a structural property. In the original equation `dP/dt = G·m·ψ − µP`, the growth
rate is itself `µ = µ_max·g(X)·ψ`. Once the culture reaches carrying capacity, `g(X) → 0` and
therefore `µ → 0`, and the protein equation loses its only loss term. `P` then rises until `ψ = 0`
and stops at `φ_crit·P_REF`, which is the same value for every dose. Over 5 h this is not visible;
over 24 h it determines the result, with 289 of 300 prior draws within 0.5% of that ceiling at 1 mM.

`k_deg` (M1.18) is the minimal correction. It is a first-order loss that lumps proteolysis,
inclusion-body partition and leakage, and it gives `P` a dose-dependent steady state,
`P_ss = G·m̃·ψ/(µ + k_deg)`, instead of a clamp.

> **Limitation.** `k_deg` is now the weakest element of the long-horizon forecast. No
> MS2-CP-specific value exists in the literature, and a 5 h time course cannot constrain it,
> because every trajectory is still rising at 300 min. Yet it determines the headline
> recommendation: at `k_deg = 0` the predicted optimum shifts *down* toward 15 µM as the horizon
> grows, whereas at `k_deg = 0.0035` it shifts *up*. The same data can therefore support opposite
> recommendations, depending on a parameter that has not been measured.

The remedy is inexpensive and follows the model → experiment → model loop: include the 12 h and
24 h time points already listed in `TOBS`. Two additional gel lanes per condition turn `k_deg`
from an assumption into a measurement. A more direct option is an arm in which translation is
halted at 300 min with an inhibitor such as chloramphenicol. With synthesis switched off, the decay
of the CP band is `k_deg`, read directly from the gel without ODE fitting. (Rifampicin is not
suitable for this purpose, because T7 RNA polymerase is rifampicin-resistant.)

### 3.2 · An emergent prediction

The right-hand panel is not monotonic. Volumetric yield peaks at an intermediate IPTG dose and then
declines, because beyond a threshold the burden term suppresses biomass faster than per-cell
expression increases. Protein per cell keeps rising; protein per litre does not.

This optimum is also computed before folding is considered. The partition model of M2 predicts a
soluble fraction that falls as the synthesis rate rises, so the optimum for *soluble* yield lies at
a lower dose still. Two independent mechanisms acting in the same direction make this a coherent and
testable design claim.

---
## 4 · Identifiability audit

### 4.1 · An exact symmetry in the cascade

The transfer function of the cascade from `f` to `m̃` is

$$H(s)=\frac{k_R}{s+k_R}\cdot\frac{k_m}{s+k_m}$$

which is exactly symmetric under `k_R ↔ k_m`: swapping the two constants gives an identical protein
trajectory.

In [ ]:
th_a = _th0.copy(); th_b = _th0.copy(); th_b[7],th_b[8] = _th0[8],_th0[7]
a,b = simulate(th_a,DOSES,t_fine), simulate(th_b,DOSES,t_fine)
print(f"max|P_a - P_b|/max(P) = {np.abs(a['P']-b['P']).max()/a['P'].max():.3e}  -> exchangeable\n")

fig,ax=plt.subplots(1,2,figsize=(9,3.2))
ax[0].plot(t_fine,a["P"][2],lw=3.5,alpha=.45,label="$k_R$=0.07, $k_m$=0.30")
ax[0].plot(t_fine,b["P"][2],"--",lw=1.6,color=INK,label="$k_R$=0.30, $k_m$=0.07")
ax[0].set(xlabel="time (min)",ylabel="CP ($\\mu$M)",title="Observable: identical")
ax[0].legend(fontsize=7)
ax[1].plot(t_fine,a["R"][2],lw=2,label="$\\tilde R$ original")
ax[1].plot(t_fine,b["R"][2],"--",lw=2,label="$\\tilde R$ swapped")
ax[1].set(xlabel="time (min)",ylabel="normalised",title="Swapped states: different")
ax[1].legend(fontsize=7); plt.tight_layout(); plt.show()

The symmetry is broken by convention. The sampler draws an ordered pair, a slow constant `k_slow`
and a ratio `ρ ∈ (0,1)` with `k_fast = k_slow/ρ`, and the slower constant is assigned to R̃ as a
labelling convention.

### 4.2 · Remaining degeneracies

| Group | Status | Resolution |
|---|---|---|
| `α_R·k_tx·N_p·k_tl` | Exactly non-identifiable | Lumped into `G` |
| `k_R ↔ k_m` | Exact symmetry | Ordering constraint (above) |
| `P_d` | Weakly identified | Transport equilibrates within minutes; needs samples at 5, 10 and 15 min |
| `eps` | Weakly identified | Needs an uninduced arm carried to late times |

---

## 5 · Comparison with measured design data

Section 3.1 argued, from the model alone, that IPTG doses above the Hill midpoint carry almost no information: the predicted spread across a 40-fold dose range was two orders of magnitude below the measurement noise. If this is a property of T7 expression and not of the particular parameterisation, an independent factorial design should rank IPTG last among its factors. The design below was run on three unrelated proteins and was not used to build the model.

The main effect of each factor is taken as the mean response at its high level minus the mean at its low level, divided by the standard deviation of the three centre-point replicates, which is the design's own estimate of run-to-run noise. A ratio near one means that the factor moved the response no further than repeating the same run would have.

In [ ]:
BB = pd.read_csv("../../data/lit_M1_induction_design.csv")
FACTORS = ["IPTG_mM", "temperature_C", "time_h"]

def main_effects(d, value="yield_mg_per_mL"):
    # the centre point is the only factor combination that is replicated; factor
    # levels differ between proteins, so it is found rather than hard-coded
    key = d.groupby(FACTORS).size()
    ctr_lv = key.idxmax()
    ctr = d[(d.IPTG_mM == ctr_lv[0]) & (d.temperature_C == ctr_lv[1]) & (d.time_h == ctr_lv[2])][value]
    sd = ctr.std(ddof=1)
    if not sd > 0: sd = np.nan      # model replicates are exact; see below
    out = {}
    for f in FACTORS:
        lv = sorted(d[f].unique())
        eff = d[d[f] == lv[-1]][value].mean() - d[d[f] == lv[0]][value].mean()
        out[f] = dict(effect=eff, ratio=abs(eff)/sd)
    return out, sd, ctr.mean()

rows = []
for prot, d in BB.groupby("protein"):
    eff, sd, m = main_effects(d)
    rows.append(dict(protein=prot, centre_mean=m, centre_sd=sd,
                     **{f"{k.split('_')[0]}_x_sd": v["ratio"] for k, v in eff.items()}))
meas = pd.DataFrame(rows)
display(meas.round(2))
print("measured ranking per protein (largest effect first):")
for _, r in meas.iterrows():
    o = r[["IPTG_x_sd", "temperature_x_sd", "time_x_sd"]].astype(float).sort_values(ascending=False)
    print(f"  {r.protein:>16} : " + " > ".join(i.replace('_x_sd','') for i in o.index) +
          f"   ({', '.join(f'{v:.1f}x' for v in o.values)})")

In [ ]:
# The same design put through the model. Temperature has no term in M1, so only
# the IPTG and time axes are predicted; the temperature column is carried through
# unchanged, which is itself the prediction (zero effect).
DESIGN = BB[BB.protein == BB.protein.iloc[0]][["run", "temperature_C", "time_h", "IPTG_mM"]]
doses_uM = np.array(sorted(DESIGN.IPTG_mM.unique()))*1e3
t_eval = np.unique(np.concatenate([[0.0], np.sort(DESIGN.time_h.unique())*60.0]))

def model_design(theta):
    s = simulate(theta, doses_uM, t_eval)
    if s is None: return None
    P = s["P"]                                        # (dose, time)
    v = []
    for _, r in DESIGN.iterrows():
        i = int(np.where(doses_uM == r.IPTG_mM*1e3)[0][0])
        j = int(np.where(t_eval == r.time_h*60.0)[0][0])
        v.append(P[i, j]*M_CP/1000.0)                 # uM -> mg/L
    return np.array(v)

TH = sample_prior(400).to_numpy()
pred = [model_design(th) for th in TH]
pred = np.array([p for p in pred if p is not None])
print(f"{len(pred)}/{len(TH)} prior draws integrated")

ratios = {"IPTG_mM": [], "time_h": [], "temperature_C": []}
for p in pred:
    d = DESIGN.copy(); d["y"] = p/np.mean(p)          # scale-free: only the ratio matters
    e, sd_m, _ = main_effects(d, value="y")
    # normalise by the MEASURED relative centre-point noise, so model and data are
    # compared on the same yardstick
    rel_noise = float(np.mean(meas.centre_sd/meas.centre_mean))
    for f in ratios: ratios[f].append(abs(e[f]["effect"])/rel_noise)

mod = pd.DataFrame({f: np.percentile(v, [5, 50, 95]) for f, v in ratios.items()},
                   index=["p5", "median", "p95"]).T
display(mod.round(2))
print(f"\nmeasured relative centre-point noise : {float(np.mean(meas.centre_sd/meas.centre_mean)):.3f}")
print("model main effect / that noise (x):")
for f in ("temperature_C", "time_h", "IPTG_mM"):
    print(f"  {f:>15} : {mod.loc[f,'median']:5.2f}  [{mod.loc[f,'p5']:.2f}, {mod.loc[f,'p95']:.2f}]")
print("\nMeasured, averaged over the three proteins:")
for f, c in (("temperature_C","temperature_x_sd"), ("time_h","time_x_sd"), ("IPTG_mM","IPTG_x_sd")):
    print(f"  {f:>15} : {meas[c].mean():5.2f}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10.4, 3.7))
labs = ["temperature", "time", "IPTG"]
keys = ["temperature_x_sd", "time_x_sd", "IPTG_x_sd"]
x = np.arange(3); w = 0.26
for k, (prot, c) in enumerate(zip(meas.protein, [NAVY, BLUE, SKY])):
    axs[0].bar(x + (k-1)*w, meas.loc[meas.protein == prot, keys].values.ravel(),
               w, color=c, label=prot)
axs[0].axhline(1.0, color=INK, lw=1, ls="--")
axs[0].text(2.35, 1.15, "run-to-run noise", fontsize=6.5, color=INK, ha="right")
axs[0].set(xticks=x, ylabel="|main effect| / centre-point SD", title="Measured (Box–Behnken, 3 proteins)")
axs[0].set_xticklabels(labs); axs[0].legend(fontsize=6.5)

mk = ["temperature_C", "time_h", "IPTG_mM"]
md_, lo_, hi_ = (np.array([mod.loc[f, c] for f in mk]) for c in ("median", "p5", "p95"))
axs[1].bar(x, md_, 0.5, color=[STEEL, MID, SKY],
           yerr=[md_-lo_, hi_-md_], capsize=4, ecolor=INK)
axs[1].axhline(1.0, color=INK, lw=1, ls="--")
axs[1].set(xticks=x, ylabel="|main effect| / measured relative noise",
           title="M1 prior predictive, same design")
axs[1].set_xticklabels(labs)
axs[1].text(0, 0.06, "not modelled", rotation=90, fontsize=7, color=INK, ha="center", va="bottom")
for a in axs: a.set_ylim(0, max(axs[0].get_ylim()[1], hi_.max()*1.15))
fig.tight_layout(); plt.show()

**What the comparison establishes, and what it does not.** The agreement is ordinal and not quantitative: the three proteins differ, the host medium differs, and the temperature range is not represented in the model, so any numerical match would be coincidental. What transfers is the ranking. Had the measured design placed IPTG first, the saturation argument of Section 3.1 would have been contradicted by the data, and the dose-ranging arm of the planned experiment would have to be widened and not narrowed. It does not, and the recommendation stands: replicates should be spent on harvest time and temperature, and IPTG above 0.1 mM should be treated as a single condition.